In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# -----------------------------
# LOAD DATASETS
# -----------------------------

dataset = pd.read_csv("dataset.csv")
precautions = pd.read_csv("symptom_precaution.csv")
severity = pd.read_csv("Symptom-severity.csv")
description = pd.read_csv("symptom_Description.csv")


# -----------------------------
# CLEAN DATASET
# -----------------------------

dataset = dataset.applymap(lambda x: x.strip() if isinstance(x, str) else x)


# -----------------------------
# CONVERT DATASET TO ML FORMAT
# -----------------------------

symptom_columns = dataset.columns[1:]

all_symptoms = set()

for col in symptom_columns:
    all_symptoms.update(dataset[col].dropna())

all_symptoms = sorted(list(all_symptoms))


df = pd.DataFrame(0, index=range(len(dataset)), columns=all_symptoms)


for i in range(len(dataset)):

    for col in symptom_columns:

        symptom = dataset.loc[i, col]

        if pd.notna(symptom):

            symptom = symptom.strip()

            if symptom in df.columns:
                df.loc[i, symptom] = 1


df["Disease"] = dataset["Disease"]


# -----------------------------
# TRAINING DATA
# -----------------------------

X = df.drop("Disease", axis=1)
y = df["Disease"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# -----------------------------
# TRAIN MODEL
# -----------------------------

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=42
)

model.fit(X_train, y_train)


# -----------------------------
# CHECK ACCURACY
# -----------------------------

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Model Accuracy:", accuracy)


# -----------------------------
# SYMPTOM LIST
# -----------------------------

symptom_list = list(X.columns)


# -----------------------------
# PREDICT DISEASE
# -----------------------------

def predict_disease(user_symptoms):

    symptom_vector = np.zeros(len(symptom_list))

    print("\nRecognized Symptoms:")

    for symptom in user_symptoms:

        symptom = symptom.strip()

        if symptom in symptom_list:

            print("✔", symptom)

            index = symptom_list.index(symptom)

            symptom_vector[index] = 1

        else:

            print("✘", symptom, "(not found in dataset)")


    symptom_vector = pd.DataFrame([symptom_vector], columns=symptom_list)

    predicted_disease = model.predict(symptom_vector)[0]

    return predicted_disease


# -----------------------------
# GET PRECAUTIONS
# -----------------------------

def get_precautions(disease):

    row = precautions[precautions["Disease"] == disease]

    if row.empty:
        return []

    return row.iloc[0,1:].dropna().values.tolist()


# -----------------------------
# GET DESCRIPTION
# -----------------------------

def get_description(disease):

    row = description[description["Disease"] == disease]

    if row.empty:
        return "No description available"

    return row.iloc[0]["Description"]


# -----------------------------
# CALCULATE SEVERITY
# -----------------------------

def calculate_severity(user_symptoms):

    score = 0

    for symptom in user_symptoms:

        symptom = symptom.strip()

        row = severity[severity["Symptom"] == symptom]

        if not row.empty:

            score += int(row["weight"].values[0])


    if score < 5:
        level = "Low"

    elif score < 10:
        level = "Medium"

    else:
        level = "High"


    return score, level


# -----------------------------
# USER INPUT
# -----------------------------

user_symptoms = [
    "high_fever",
    "headache",
    "joint_pain",
    "vomiting"
]




# -----------------------------
# PREDICTION PIPELINE
# -----------------------------

disease = predict_disease(user_symptoms)

prec = get_precautions(disease)

desc = get_description(disease)

severity_score, severity_level = calculate_severity(user_symptoms)


# -----------------------------
# FINAL OUTPUT
# -----------------------------

print("\nPredicted Disease:", disease)

print("\nDescription:")
print(desc)

print("\nSeverity Level:", severity_level)

print("\nPrecautions:")

for p in prec:
    print("-", p)

C:\Users\nikhi\AppData\Local\Temp\ipykernel_4976\3128616934.py:22: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  dataset = dataset.applymap(lambda x: x.strip() if isinstance(x, str) else x)


Model Accuracy: 1.0

Recognized Symptoms:
✔ high_fever
✔ headache
✔ joint_pain
✔ vomiting

Predicted Disease: Paralysis (brain hemorrhage)

Description:
Intracerebral hemorrhage (ICH) is when blood suddenly bursts into brain tissue, causing damage to your brain. Symptoms usually appear suddenly during ICH. They include headache, weakness, confusion, and paralysis, particularly on one side of your body.

Severity Level: High

Precautions:
- massage
- eat healthy
- exercise
- consult doctor
